# Project: BBLF AI Selector v2
# Section: Optimisation & Simulation Tool
# Sub Section: During Tourny (Post Rnd 1)

Goal: Select the optimal starting 12 players prior to the start of the tournament 

Things to add:
1. Additional constraints to capture fantasy nuances e.g. captain points, vice captain trick 


# 0. Prerequistes

In [1]:
# pip install --upgrade pip

In [2]:
# !pip install mip==1.16rc0
!pip install mip

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
# 0. Prerequistes

import pandas as pd
import numpy as np
import os
import random
from mip import Model, xsum, maximize, BINARY 
from scipy.stats import norm
import joblib
from concurrent.futures import ProcessPoolExecutor
import warnings
warnings.filterwarnings("ignore")

# Import intermediary functions
from Optimisation_Functions import roll_rnd_price_fn, optimise_fn_efp, optimise_fn_sim_fp, optimise_setup_fn

# Set working directory
os.getcwd()
directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2'
add_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/post_round_1/'
over_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/overall'
py_data_score_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/python_data/scoring'

In [4]:
# Optimisation Strategy
opt_strat = 'efp' # Options: 'efp' (new player expected fantasy points appraoch), 'sim' (new player fantasy point distribution approach)
current_rnd = 2 # Current Round of the Tournament
exp_pts_model = 'all' # Options: 'bat_bowl' (batting + bowling model), 'all' (combined model)

# Constraints
squad_players = 15

# For Simulation Optimisation Strategy
conf_int = 0.2
sim_num = 100

# 1. New EFP Approach

# a. Data Extraction 

In [5]:
# New Season Optimisation Strategy (Expected Fantasy Points)
if opt_strat == 'efp':
    
    # Data Extraction 
    # Pull in player price data csv file 
    price_df = pd.read_csv(os.path.join(add_data_directory,'player_price_2026_efp_rnd_1.csv'), low_memory=False)

    # Create Player Role Flags
    price_df["Wk_f"] = np.where((price_df["Role"] == "WK"), 1, 0)
    price_df["Bat_f"] = np.where((price_df["Role"] == "WK") |(price_df["Role"] == "BAT") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df["Bowl_f"] = np.where((price_df["Role"] == "BOWL") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df = price_df[["Full_Name", "Price", "Team","Wk_f", "Bat_f", "Bowl_f", "Role", "Available", "In_Team"]].rename(columns = {"Full_Name": "Name"}) 

    team_fix_df = pd.read_csv(os.path.join(over_data_directory,'team_loc_fixture.csv'), low_memory=False)
    # team_fix_df = team_fix_df[team_fix_df.Round >= current_rnd].dropna()

    # Join team fixture to player price to create a row for each game
    game_df = pd.merge(price_df , team_fix_df, left_on = ["Team"], right_on = ["Team"], how = "left")

    # Pull player expected points csv file
    if exp_pts_model == 'bat_bowl':
        efp_df = pd.read_csv(os.path.join(py_data_score_directory,'bbl15_efp_bat_bowl_model_score_short_pr1.csv'), low_memory=False)
    elif exp_pts_model == 'all':
        efp_df = pd.read_csv(os.path.join(py_data_score_directory,'bbl15_efp_model_score_short_pr1.csv'), low_memory=False)

    # Join on expected points for each game
    player_df_raw = pd.merge(game_df, efp_df, left_on = ["Name", "Team", "Opposition", "Venue", "Home_f","Round"], right_on = ["Name", "Team", "opp", "venue", "Home_f","Round"], how = "left")
    # If pts_type is exp then add fielding points to expected points
    player_df_raw["exp_points"] = np.where((player_df_raw["Role"] == "WK") & (player_df_raw["pts_type"] == "exp"), player_df_raw["exp_pts"] + 11.68,
                                           np.where((player_df_raw["pts_type"] == "exp"), player_df_raw["exp_pts"] + 4.42, player_df_raw["exp_pts"]))
    
    # Drop unnecessary columns
    player_df_raw = player_df_raw[["Name", "Team", "Role", "Price", "Wk_f", "Bat_f", "Bowl_f", "Available", "In_Team", "Round", "opp", "venue", "Home_f", "exp_points"]]
    player_df_raw["weight"] = 1

    # Create game count per player
    player_df_raw["game_num"] = player_df_raw.groupby("Name").cumcount() + 1
    
else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

# b. Player price prediction for each round & data prep for optimisation

In [6]:
# # Copy player init df to avoid modifying original
# player_df_init_2 = player_df_init.copy()

# # a. Derive Features required for Pricing Models
# # Aggregate by player name per round
# player_df = player_df_init_2.groupby(['Name', 'Price', "Team", "Round", "Wk_f", "Bat_f", "Bowl_f", "Role", "weight","Available", "In_Team"], as_index=False).agg(
# exp_rnd_points=('exp_points',"sum"),
# games_in_round=('Round',"count"))

# # Create game_num variable
# pts_per_game = player_df_init_2[['Name', 'game_num', 'exp_points']].sort_values(['Name', 'game_num'])

# # Vectorized feature calculation using groupby and transform
# grouped = pts_per_game.groupby('Name')['exp_points']

# # Calculate all rolling features at once
# pts_per_game['curr_game_pts'] = pts_per_game['exp_points']
# pts_per_game['prev_game_pts'] = grouped.shift(1)
# pts_per_game['two_prev_game_pts'] = grouped.shift(2)
# pts_per_game['last_2_games_ma_pts'] = grouped.transform(lambda x: x.rolling(2, min_periods=1).mean())
# pts_per_game['last_3_games_ma_pts'] = grouped.transform(lambda x: x.rolling(3, min_periods=1).mean())
# pts_per_game['seas_avg_games_pts'] = grouped.transform(lambda x: x.expanding().mean())

# # Drop the original exp_points column and keep calculated features
# bbl15_game_pts_table_pre = pts_per_game.drop('exp_points', axis=1)

# # Add team and round info
# player_team = player_df_init_2[['Name', 'Team']].drop_duplicates()
# game_rnd_team_df = player_df_init_2[['Team','Round','game_num']].drop_duplicates()

# bbl15_game_pts_table = bbl15_game_pts_table_pre.merge(player_team, on='Name').merge(game_rnd_team_df, on=['Team', 'game_num']).drop('Team', axis=1)

# # For player double gameweek rounds, only return the second game row
# # Keep only max game_num per player per round
# bbl15_game_pts_table = bbl15_game_pts_table.loc[
#     bbl15_game_pts_table.groupby(['Name', 'Round'])['game_num'].idxmax()
# ]

# bbl15_game_pts_table['Round'] = bbl15_game_pts_table['Round'].astype("Int64")
# bbl15_game_pts_table = bbl15_game_pts_table.sort_values(['Name','Round'])

# # Add player price
# bbl15_game_pts_table = pd.merge(bbl15_game_pts_table, price_df[['Name', 'Price']], on = 'Name', how = 'left').rename(columns={"Price":"price_pre"})

# # Price prediction - OPTIMIZED with bulk predictions
# # player_df_lags = bbl15_game_pts_table[['Name', 'Round', 'game_num', 'price_pre', 'seas_avg_games_pts', 'last_2_games_ma_pts', 'last_3_games_ma_pts']]

# player_df_lags = bbl15_game_pts_table[['Name', 'Round', 'game_num', 'price_pre', 'seas_avg_games_pts', 'last_2_games_ma_pts', 'last_3_games_ma_pts']].copy()

# # Drop previous rounds as only required for feature calculation
# player_df_lags = player_df_lags[player_df_lags['Round'] >= current_rnd]

# # Pre-compute all predictions in bulk
# player_df_lags['Price_Pred'] = np.nan

# # Loop per player in order of game_num, updating next game's price_pre with the prediction
# for name, grp in player_df_lags.groupby('Name'):
#     grp_sorted = grp.sort_values('game_num')
#     if grp_sorted.empty:
#         continue
#     for i, idx in enumerate(grp_sorted.index):
#         row = player_df_lags.loc[idx]
#         gnum = int(row['game_num']) if not pd.isna(row['game_num']) else 1

#         try:
#             if gnum == 1:
#                 X = np.array([[row['price_pre'], row['seas_avg_games_pts']]])
#                 pred = price_model_obj_1.predict(X)[0]
#             elif gnum == 2:
#                 X = np.array([[row['price_pre'], row['last_2_games_ma_pts']]])
#                 pred = price_model_obj_2.predict(X)[0]
#             else:
#                 X = np.array([[row['price_pre'], row['last_3_games_ma_pts']]])
#                 pred = price_model_obj_3.predict(X)[0]
#         except Exception:
#             # Fallback: if model prediction fails, keep original price_pre
#             pred = row['price_pre']

#         player_df_lags.at[idx, 'Price_Pred'] = pred

#         # If there is a next game for this player, set its price_pre to this prediction
#         if i + 1 < len(grp_sorted):
#             next_idx = grp_sorted.index[i + 1]
#             player_df_lags.at[next_idx, 'price_pre'] = pred
            
# # b. Build price dataframe with rolling predictions
# player_df_new_list = []

# for player in player_df['Name'].unique():
#     player_lags = player_df_lags[player_df_lags['Name'] == player].sort_values('Round')
#     player_rounds = player_lags['Round'].values
    
#     # Current round - use actual price
#     curr_mask = player_df['Name'] == player
#     curr_data = player_df[curr_mask & (player_df['Round'] == current_rnd)].copy()
#     if not curr_data.empty:
#         player_df_new_list.append(curr_data)
#         last_known_price = curr_data['Price'].values[0]
#     else:
#         # If no current round data, get price from player_df
#         last_known_price = player_df[curr_mask]['Price'].iloc[0] if not player_df[curr_mask].empty else None
    
#     # Future rounds - use predicted prices only if player was available in current round
#     for i, rnd in enumerate(player_rounds[:-1]):
#         next_rnd = player_rounds[i + 1]
        
#         future_data = player_df[curr_mask & (player_df['Round'] == next_rnd)].copy()
#         if not future_data.empty:
#             # Check if player was available in the current round (rnd)
#             curr_rnd_data = player_df[curr_mask & (player_df['Round'] == rnd)]
#             if not curr_rnd_data.empty and curr_rnd_data['Available'].values[0] == 1:
#                 # Player was available - apply predicted price change
#                 pred_price = player_lags[player_lags['Round'] == rnd]['Price_Pred'].values[0]
#                 future_data['Price'] = pred_price
#                 last_known_price = pred_price
#             else:
#                 # Player was not available - keep last known price (no change)
#                 future_data['Price'] = last_known_price
#             player_df_new_list.append(future_data)

# if player_df_new_list:
#     player_df = pd.concat(player_df_new_list, ignore_index=True)

# else:
#     player_df = pd.DataFrame()

# # c. Prepare Player DataFrames for optimisation
# # Create additional rows for players who do not play in every round
# all_rounds = list(range(int(player_df['Round'].min()), int(player_df['Round'].max()) + 1))
# all_players = player_df['Name'].unique()
# full_index = pd.MultiIndex.from_product([all_players, all_rounds], names=['Name', 'Round'])
# player_df = player_df.set_index(['Name', 'Round']).reindex(full_index).reset_index()
# player_df['Price'] = player_df['Price'].ffill()
# player_df['Team'] = player_df['Team'].ffill()
# player_df['Wk_f'] = player_df['Wk_f'].ffill()
# player_df['Bat_f'] = player_df['Bat_f'].ffill()
# player_df['Bowl_f'] = player_df['Bowl_f'].ffill()
# player_df['Role'] = player_df['Role'].ffill()
# player_df['weight'] = player_df['weight'].ffill()
# player_df['exp_rnd_points'] = player_df['exp_rnd_points'].fillna(-100)
# player_df['games_in_round'] = player_df['games_in_round'].fillna(0)
# player_df['Available'] = np.where(player_df['exp_rnd_points'] == -100, 0, player_df['Available'])
# player_df['In_Team'] = player_df['In_Team'].ffill()

# # Split player df by round
# player_dfs = {i: player_df[player_df['Round'] == i].reset_index(drop=True) for i in range(1, 10)}
# player_df_r1, player_df_r2, player_df_r3 = player_dfs[1], player_dfs[2], player_dfs[3]
# player_df_r4, player_df_r5, player_df_r6 = player_dfs[4], player_dfs[5], player_dfs[6]
# player_df_r7, player_df_r8, player_df_r9 = player_dfs[7], player_dfs[8], player_dfs[9]

# # Drop intermediate variables to free memory
# del player_df_init_2, player_df, pts_per_game, bbl15_game_pts_table_pre, bbl15_game_pts_table, player_team, game_rnd_team_df, player_df_lags, player_df_new_list

# # Return player dfs for each round
# return player_df_r1, player_df_r2, player_df_r3, player_df_r4, player_df_r5, player_df_r6, player_df_r7, player_df_r8, player_df_r9


In [7]:
if opt_strat == 'efp':
    # Incorporate Player Actual Points into dataframes used for pricing model

    
    # Import Pricing Models
    # Load Model Objects
    price_model_obj_1 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_1_game'))
    price_model_obj_2 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_2_game'))
    price_model_obj_3 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_3_game'))

    # Create rolling round player price dataframes
    player_df_r1, player_df_r2, player_df_r3, player_df_r4, player_df_r5, player_df_r6, player_df_r7, player_df_r8, player_df_r9 = roll_rnd_price_fn(player_df_raw, price_df, current_rnd, price_model_obj_1, price_model_obj_2, price_model_obj_3)

else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

# c. Optimisation Process (Pre Tournament Selection) - Starting 12 Players

Optimisation Objective: Maximise the number of expected fantasy points

Constraints: 
1. Number of players selected must be 12 + x bench player
2. Atleast 1 wicketkeeper
3. Atleast 6 batters
4. Atleast 5 bowlers
5. Total budget of team is less than $1,802,500 (As bench costs $197,500)

## Optimisation Setup

In [ ]:
# EFP Whole Tournament Optimisation
if opt_strat == 'efp':
    # a. EFP Optimisation Variables Setup
    points_r1, price_r1, weight_r1, in_team_r1, available_r1, wk_weight_r1, bat_weight_r1, bowl_weight_r1, \
        play_cnt_r1, total_player_r1, wk_cnt_r1, total_wk_r1, bat_cnt_r1, total_bat_r1, bowl_cnt_r1, total_bowl_r1, \
        budget_r1, total_budget_r1, player_df_r1, cnt_r1, max_player_r1, \
        points_r2, price_r2, weight_r2, in_team_r2, available_r2, wk_weight_r2, bat_weight_r2, bowl_weight_r2, \
        play_cnt_r2, total_player_r2, wk_cnt_r2, total_wk_r2, bat_cnt_r2, total_bat_r2, bowl_cnt_r2, total_bowl_r2, \
        budget_r2, total_budget_r2, team_play_cnt_r2, total_team_player_r2, player_df_r2, cnt_r2, max_player_r2, \
        points_r3, price_r3, weight_r3, in_team_r3, available_r3, wk_weight_r3, bat_weight_r3, bowl_weight_r3, \
        play_cnt_r3, total_player_r3, wk_cnt_r3, total_wk_r3, bat_cnt_r3, total_bat_r3, bowl_cnt_r3, total_bowl_r3, \
        budget_r3, total_budget_r3, team_play_cnt_r3, total_team_player_r3, player_df_r3, cnt_r3, max_player_r3, \
        points_r4, price_r4, weight_r4, in_team_r4, available_r4, wk_weight_r4, bat_weight_r4, bowl_weight_r4, \
        play_cnt_r4, total_player_r4, wk_cnt_r4, total_wk_r4, bat_cnt_r4, total_bat_r4, bowl_cnt_r4, total_bowl_r4, \
        budget_r4, total_budget_r4, team_play_cnt_r4, total_team_player_r4, player_df_r4, cnt_r4, max_player_r4, \
        points_r5, price_r5, weight_r5, in_team_r5, available_r5, wk_weight_r5, bat_weight_r5, bowl_weight_r5, \
        play_cnt_r5, total_player_r5, wk_cnt_r5, total_wk_r5, bat_cnt_r5, total_bat_r5, bowl_cnt_r5, total_bowl_r5, \
        budget_r5, total_budget_r5, team_play_cnt_r5, total_team_player_r5, player_df_r5, cnt_r5, max_player_r5, \
        points_r6, price_r6, weight_r6, in_team_r6, available_r6, wk_weight_r6, bat_weight_r6, bowl_weight_r6, \
        play_cnt_r6, total_player_r6, wk_cnt_r6, total_wk_r6, bat_cnt_r6, total_bat_r6, bowl_cnt_r6, total_bowl_r6, \
        budget_r6, total_budget_r6, team_play_cnt_r6, total_team_player_r6, player_df_r6, cnt_r6, max_player_r6, \
        points_r7, price_r7, weight_r7, in_team_r7, available_r7, wk_weight_r7, bat_weight_r7, bowl_weight_r7, \
        play_cnt_r7, total_player_r7, wk_cnt_r7, total_wk_r7, bat_cnt_r7, total_bat_r7, bowl_cnt_r7, total_bowl_r7, \
        budget_r7, total_budget_r7, team_play_cnt_r7, total_team_player_r7, player_df_r7, cnt_r7, max_player_r7, \
        points_r8, price_r8, weight_r8, in_team_r8, available_r8, wk_weight_r8, bat_weight_r8, bowl_weight_r8, \
        play_cnt_r8, total_player_r8, wk_cnt_r8, total_wk_r8, bat_cnt_r8, total_bat_r8, bowl_cnt_r8, total_bowl_r8, \
        budget_r8, total_budget_r8, team_play_cnt_r8, total_team_player_r8, player_df_r8, cnt_r8, max_player_r8, \
        points_r9, price_r9, weight_r9, in_team_r9, available_r9, wk_weight_r9, bat_weight_r9, bowl_weight_r9, \
        play_cnt_r9, total_player_r9, wk_cnt_r9, total_wk_r9, bat_cnt_r9, total_bat_r9, bowl_cnt_r9, total_bowl_r9, \
        budget_r9, total_budget_r9, team_play_cnt_r9, total_team_player_r9, player_df_r9, cnt_r9, max_player_r9 =  optimise_setup_fn(
            player_df_r1, player_df_r2, player_df_r3, player_df_r4, player_df_r5, player_df_r6, player_df_r7, player_df_r8, player_df_r9,squad_players)

else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

## Optimisation Runner

In [ ]:
# EFP Whole Tournament Optimisation
if opt_strat == 'efp':
    sel_player_df, sel_player_df_r1, sel_player_df_r2, sel_player_df_r3, sel_player_df_r4, sel_player_df_r5, sel_player_df_r6, sel_player_df_r7,sel_player_df_r8, sel_player_df_r9 = optimise_fn_efp(
        # Round 1
        points_r1, price_r1, weight_r1, in_team_r1, available_r1, wk_weight_r1, bat_weight_r1, bowl_weight_r1,
        play_cnt_r1, total_player_r1, wk_cnt_r1, total_wk_r1, bat_cnt_r1, total_bat_r1, bowl_cnt_r1, total_bowl_r1,
        budget_r1, total_budget_r1, player_df_r1, cnt_r1, max_player_r1,
        # Round 2
        points_r2, price_r2, weight_r2, in_team_r2, available_r2, wk_weight_r2, bat_weight_r2, bowl_weight_r2,
        play_cnt_r2, total_player_r2, wk_cnt_r2, total_wk_r2, bat_cnt_r2, total_bat_r2, bowl_cnt_r2, total_bowl_r2,
        budget_r2, total_budget_r2, team_play_cnt_r2, total_team_player_r2, player_df_r2, cnt_r2, max_player_r2,
        # Round 3
        points_r3, price_r3, weight_r3, in_team_r3, available_r3, wk_weight_r3, bat_weight_r3, bowl_weight_r3,
        play_cnt_r3, total_player_r3, wk_cnt_r3, total_wk_r3, bat_cnt_r3, total_bat_r3, bowl_cnt_r3, total_bowl_r3,
        budget_r3, total_budget_r3, team_play_cnt_r3, total_team_player_r3, player_df_r3, cnt_r3, max_player_r3,
        # Round 4
        points_r4, price_r4, weight_r4, in_team_r4, available_r4, wk_weight_r4, bat_weight_r4, bowl_weight_r4,
        play_cnt_r4, total_player_r4, wk_cnt_r4, total_wk_r4, bat_cnt_r4, total_bat_r4, bowl_cnt_r4, total_bowl_r4,
        budget_r4, total_budget_r4, team_play_cnt_r4, total_team_player_r4, player_df_r4, cnt_r4, max_player_r4,
        # Round 5
        points_r5, price_r5, weight_r5, in_team_r5, available_r5, wk_weight_r5, bat_weight_r5, bowl_weight_r5,
        play_cnt_r5, total_player_r5, wk_cnt_r5, total_wk_r5, bat_cnt_r5, total_bat_r5, bowl_cnt_r5, total_bowl_r5,
        budget_r5, total_budget_r5, team_play_cnt_r5, total_team_player_r5, player_df_r5, cnt_r5, max_player_r5,
        # Round 6
        points_r6, price_r6, weight_r6, in_team_r6, available_r6, wk_weight_r6, bat_weight_r6, bowl_weight_r6,
        play_cnt_r6, total_player_r6, wk_cnt_r6, total_wk_r6, bat_cnt_r6, total_bat_r6, bowl_cnt_r6, total_bowl_r6,
        budget_r6, total_budget_r6, team_play_cnt_r6, total_team_player_r6, player_df_r6, cnt_r6, max_player_r6,
        # Round 7
        points_r7, price_r7, weight_r7, in_team_r7, available_r7, wk_weight_r7, bat_weight_r7, bowl_weight_r7,
        play_cnt_r7, total_player_r7, wk_cnt_r7, total_wk_r7, bat_cnt_r7, total_bat_r7, bowl_cnt_r7, total_bowl_r7,
        budget_r7, total_budget_r7, team_play_cnt_r7, total_team_player_r7, player_df_r7, cnt_r7, max_player_r7,
        # Round 8
        points_r8, price_r8, weight_r8, in_team_r8, available_r8, wk_weight_r8, bat_weight_r8, bowl_weight_r8,
        play_cnt_r8, total_player_r8, wk_cnt_r8, total_wk_r8, bat_cnt_r8, total_bat_r8, bowl_cnt_r8, total_bowl_r8,
        budget_r8, total_budget_r8, team_play_cnt_r8, total_team_player_r8, player_df_r8, cnt_r8, max_player_r8,
        # Round 9
        points_r9, price_r9, weight_r9, in_team_r9, available_r9, wk_weight_r9, bat_weight_r9, bowl_weight_r9,
        play_cnt_r9, total_player_r9, wk_cnt_r9, total_wk_r9, bat_cnt_r9, total_bat_r9, bowl_cnt_r9, total_bowl_r9,
        budget_r9, total_budget_r9, team_play_cnt_r9, total_team_player_r9, player_df_r9, cnt_r9, max_player_r9
    )

else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

## Optimisation Results

In [ ]:
# EFP Whole Tournament Optimisation
if opt_strat == 'efp':
    # Display Round-wise Selected Teams
    print("Round 1 Selected Team")
    display(sel_player_df_r1.sort_values(by='exp_rnd_points', ascending=False))

    # Save Optimal Team to CSV
    if exp_pts_model == 'all':
        # sel_player_df.to_csv(os.path.join(add_data_directory,'optim/EFP_optimal_pre_tourny_team.csv'), index=False)
        # sel_player_df_r1.to_csv(os.path.join(add_data_directory,'optim/EFP_optimal_pre_tourny_team_rnd.csv'), index=False)
        print("EFP Optimal Team Saved")
    elif exp_pts_model == 'bat_bowl':
        sel_player_df.to_csv(os.path.join(add_data_directory,'optim/EFP_optimal_pre_tourny_team_dual.csv'), index=False)
        sel_player_df_r1.to_csv(os.path.join(add_data_directory,'optim/EFP_optimal_pre_tourny_team_dual_rnd.csv'), index=False)
        print("EFP Bat + Bowl Optimal Team Saved")
else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

# 2. New Distribution Approach

# a. Data Extraction 

In [ ]:
# New Optimisation Strategy (Player Fantasy Point Distribution)
if opt_strat == 'sim':
    # Data Extraction 
    # Pull in player price data csv file 
    price_df = pd.read_csv(os.path.join(add_data_directory,'player_price_2026_sim_rnd_1.csv'), low_memory=False)

    # Player Role Flags
    price_df["Wk_f"] = np.where((price_df["Role"] == "WK"), 1, 0)
    price_df["Bat_f"] = np.where((price_df["Role"] == "WK") |(price_df["Role"] == "BAT") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df["Bowl_f"] = np.where((price_df["Role"] == "BOWL") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df = price_df[["Full_Name", "Price", "Team","Wk_f", "Bat_f", "Bowl_f", "Role", "Available", "In_Team"]].rename(columns = {"Full_Name": "Name"}) 

    team_fix_df = pd.read_csv(os.path.join(over_data_directory,'team_loc_fixture_upd.csv'), low_memory=False)
    team_fix_df = team_fix_df[team_fix_df.Round >= current_rnd].dropna()

    # Join team fixture to player price to create a row for each game
    game_df = pd.merge(price_df, team_fix_df, left_on = ["Team"], right_on = ["Team"], how = "left")

    # Pull player expected & std points csv file
    if exp_pts_model == 'bat_bowl':
        efp_raw_df = pd.read_csv(os.path.join(py_data_score_directory,'bbl15_efp_bat_bowl_model_score_short_pr1.csv'), low_memory=False)
        sfp_df = pd.read_csv(os.path.join(py_data_score_directory,'bbl15_sfp_model_score_pre_tourny_short.csv'), low_memory=False)
        efp_df = efp_raw_df.merge(sfp_df, left_on = ["player", "Name", "Team", "opp", "venue", "Home_f","Round"], right_on = ["player", "Name", "Team", "opp", "venue", "Home_f","Round"], how = "left")

    elif exp_pts_model == 'all':
        efp_raw_df = pd.read_csv(os.path.join(py_data_score_directory,'bbl15_efp_model_score_short_pr1.csv'), low_memory=False)
        sfp_df = pd.read_csv(os.path.join(py_data_score_directory,'bbl15_sfp_model_score_pre_tourny_short.csv'), low_memory=False)
        efp_df = efp_raw_df.merge(sfp_df, left_on = ["player", "Name", "Team", "opp", "venue", "Home_f","Round"], right_on = ["player", "Name", "Team", "opp", "venue", "Home_f","Round"], how = "left")

    # # Join on expected points for each game
    player_df_raw = pd.merge(game_df, efp_df, left_on = ["Name", "Team", "Opposition", "Venue", "Home_f","Round"], right_on = ["Name", "Team", "opp", "venue", "Home_f","Round"], how = "left")
    player_df_raw["exp_pts"] = np.where((player_df_raw["Role"] == "WK"), player_df_raw["exp_pts"] + 11.68, player_df_raw["exp_pts"] + 4.42)
    player_df_raw = player_df_raw.rename(columns={"exp_pts": "mean"})
    player_df_raw = player_df_raw.rename(columns={"sd_pts": "std_dev"})

    player_df_raw["weight"] = 1
    
    # Create game count per player
    player_df_raw["game_num"] = player_df_raw.groupby("Name").cumcount() + 1

else:
    print("Change opt_strat to sim for player fp distribution Optimisation Strategy")


# b. Optimisation Process (Pre Tournament Selection) - Starting 12 Players

Optimisation Objective: Maximise the number of expected fantasy points

Constraints: 
1. Number of players selected must be 13 (Incl Bench Player)
2. Atleast 1 wicketkeeper
3. Atleast 6 batters
4. Atleast 5 bowlers
5. Total starting budget of team is less than $1,802,500 (As bench costs $197,500)

In [ ]:
# Sim FP Whole Tournament Optimisation
if opt_strat == 'sim':
    # Load Model Objects
    price_model_obj_1 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_1_game'))
    price_model_obj_2 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_2_game'))
    price_model_obj_3 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_3_game'))

    # Run Simulation Optimisation Function (with parallel execution enabled)
    all_sim_sel_players = optimise_fn_sim_fp(
        conf_int, sim_num, current_rnd, player_df_raw, price_df, 
        price_model_obj_1, price_model_obj_2, price_model_obj_3,
        squad_players, use_parallel=True  # Set to False for sequential execution
    )

else:
    print("Change opt_strat to sim for player fp distribution Optimisation Strategy")
 

In [ ]:
# Find Most Common Player Combinations Across Simulations
if opt_strat == 'sim':
    from collections import Counter
    
    # Group by simulation and create player combination sets
    sim_combinations = []
    
    for sim_id in all_sim_sel_players['Simulation'].unique():
        sim_players = all_sim_sel_players[all_sim_sel_players['Simulation'] == sim_id]['Name'].unique()
        # Create a frozenset to make it hashable for counting
        player_set = frozenset(sim_players)
        sim_combinations.append(player_set)
    
    # Count frequency of each combination
    combination_counts = Counter(sim_combinations)
    
    # Sort by frequency (descending)
    sorted_combinations = sorted(combination_counts.items(), key=lambda x: x[1], reverse=True)
    
    # Display results
    print(f"Total Unique Player Combinations: {len(combination_counts)}")
    print(f"Total Simulations: {len(sim_combinations)}\n")
    
    # Show top 10 combinations
    print("=" * 80)
    print("TOP 10 MOST COMMON PLAYER COMBINATIONS")
    print("=" * 80)
    
    for rank, (player_combo, count) in enumerate(sorted_combinations[:10], 1):
        percentage = (count / len(sim_combinations)) * 100
        print(f"\nRank {rank}: {count}/{len(sim_combinations)} simulations ({percentage:.1f}%)")
        print(f"Players: {sorted(list(player_combo))}")
        print("-" * 80)

else:
    print("Change opt_strat to sim for player fp distribution Optimisation Strategy")

In [ ]:
# Create DataFrame Summary of Most Common Combinations
if opt_strat == 'sim':
    # Create a dataframe with combination rankings
    combination_summary = []
    
    for rank, (player_combo, count) in enumerate(sorted_combinations, 1):
        percentage = (count / len(sim_combinations)) * 100
        combination_summary.append({
            'Rank': rank,
            'Frequency': count,
            'Percentage': f"{percentage:.1f}%",
            'Players': ', '.join(sorted(list(player_combo))),
            'Num_Players': len(player_combo)
        })
    
    combination_summary_df = pd.DataFrame(combination_summary)
    
    # Display the top combinations as a nice table
    print("\nCOMBINATION SUMMARY TABLE:")
    print(combination_summary_df.head(10).to_string(index=False))
    
    # Get the most common combination
    if sorted_combinations:
        most_common_players = sorted(list(sorted_combinations[0][0]))
        most_common_count = sorted_combinations[0][1]
        most_common_percentage = (most_common_count / len(sim_combinations)) * 100
        
        print(f"\n{'='*80}")
        print(f"MOST COMMON COMBINATION SELECTED:")
        print(f"{'='*80}")
        print(f"Appears in: {most_common_count}/{len(sim_combinations)} simulations ({most_common_percentage:.1f}%)")
        print(f"\nPlayers ({len(most_common_players)}):")
        for i, player in enumerate(most_common_players, 1):
            print(f"  {i:2d}. {player}")
        print(f"{'='*80}")

    # Format most common players into dataframe
    most_common_players = pd.DataFrame(most_common_players, columns=['Name'])

    # Merge with player details
    most_common_players = pd.merge(most_common_players, price_df, on='Name', how='left')
    
    # Save Combination Summary to CSV
    if exp_pts_model == 'all':
        # most_common_players.to_csv(os.path.join(add_data_directory,'optim/SFP_optimal_pre_tourny_team_12.csv'), index=False)
        # combination_summary_df.head(5).to_csv(os.path.join(add_data_directory,'optim/SFP_optimal_pre_tourny_combos_12.csv'), index=False)

        print("Sim FP Combination Summary Saved")
    elif exp_pts_model == 'bat_bowl':
        most_common_players.to_csv(os.path.join(add_data_directory,'optim/SFP_optimal_pre_tourny_team_dual.csv'), index=False)
        combination_summary_df.head(5).to_csv(os.path.join(add_data_directory,'optim/SFP_optimal_pre_tourny_combos.csv'), index=False)

        print("Sim FP Combination Summary Saved")

else:
    print("Change opt_strat to sim for player fp distribution Optimisation Strategy")